In [62]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import f1_score, make_scorer

df = pd.read_csv("Data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

# 1. Tenure Group
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

# 2. Spend Per Month
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["spend_per_month"] = (
    df["TotalCharges"] / df["tenure"].replace(0, np.nan)
).fillna(df["MonthlyCharges"])

# 3. Total Services
service_columns = [
    "PhoneService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["total_services"] = (
    (df[service_columns] == "Yes").sum(axis=1)
    + (df["InternetService"] != "No").astype(int)
)

# 4. Has Family
df["has_family"] = (
    (df["Partner"] == "Yes") |
    (df["Dependents"] == "Yes")
).astype(int)

# Display the 4 new features
df[[
    "tenure_group",
    "spend_per_month",
    "total_services",
    "has_family"
]].head()

,tenure_group,spend_per_month,total_services,has_family
0,0-12,29.850000,2,1
1,25-48,55.573529,4,0
2,0-12,54.075000,4,0
3,25-48,40.905556,4,0
4,0-12,75.825000,2,0


In [63]:
# Separate input and target
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

# Identify numerical columns
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "spend_per_month",
    "total_services",
    "has_family"
]

# Identify categorical columns
categorical_features = [
    col for col in X.columns if col not in numeric_features
]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessors
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [64]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [65]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

In [66]:
scoring = {
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, pos_label="Yes")
}

In [67]:
logistic_scores = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

rf_scores = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

In [68]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "CV ROC-AUC": [
        logistic_scores["test_roc_auc"].mean(),
        rf_scores["test_roc_auc"].mean()
    ],
    "CV F1": [
        logistic_scores["test_f1"].mean(),
        rf_scores["test_f1"].mean()
    ]
})

results = results.round(3)

results

,Model,CV ROC-AUC,CV F1
0,Logistic Regression,0.846,0.594
1,Random Forest,0.823,0.550


In [69]:
results.to_csv("results.csv", index=False)